# Brief 04 : Est-ce que ce prêt doit être accordé ou rejetté ?

In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("data/SBAnational.csv")
df.columns

/tmp/ipykernel_41962/992082409.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/SBAnational.csv")


Index(['LoanNr_ChkDgt', 'Name', 'City', 'State', 'Zip', 'Bank', 'BankState',
       'NAICS', 'ApprovalDate', 'ApprovalFY', 'Term', 'NoEmp', 'NewExist',
       'CreateJob', 'RetainedJob', 'FranchiseCode', 'UrbanRural', 'RevLineCr',
       'LowDoc', 'ChgOffDate', 'DisbursementDate', 'DisbursementGross',
       'BalanceGross', 'MIS_Status', 'ChgOffPrinGr', 'GrAppv', 'SBA_Appv'],
      dtype='object')

In [3]:
# Afficher le DataFrame nettoyé
df.shape[0]

899164

# Explication des colonnes

### Identification et localisation de l'entreprise

- **LoanNr_ChkDgt** : Numéro unique du prêt   <span style="color:red;">Useless</span>
- **Name** : Nom de l'entreprise bénéficiaire du prêt. <span style="color:red;">Useless</span>
- **City** : Ville où est située l'entreprise.  <span style="color:grey;">Redondant</span>
- **State** : État    <span style="color:green;">OK</span>
- **Zip** : Code postal   <span style="color:grey;">Redondant</span>

# Informations sur les prêts aux entreprises

## Informations sur la banque prêteuse

- **Bank** : Nom de la banque ayant accordé le prêt.  <span style="color:red;">Trop de valeurs uniques</span>
- **BankState** : État où est située la banque prêteuse.  <span style="color:green;">OK</span>

## Informations sur l'entreprise

- **NAICS** : Code de classification de l'industrie.   <span style="color:green;">OK</span>  -200.000 valeurs si not null notNaN
- **NoEmp** : Nombre d'employés de l'entreprise au moment de la demande de prêt.  <span style="color:green;">OK</span>
- **NewExist** : Indique si l'entreprise est nouvelle (**1**) ou existante (**2**). <span style="color:green;">OK</span> 

## Détails du prêt

- **ApprovalDate** : Date à laquelle le prêt a été approuvé.  <span style="color:grey;">redondant</span>
- **ApprovalFY** : Année fiscale d'approbation du prêt <span style="color:green;">OK</span>
- **Term** : Durée du prêt en mois.  <span style="color:green;">OK</span>
- **CreateJob** : Nombre d'emplois que l'entreprise prévoit de créer grâce au prêt.  <span style="color:green;">OK</span>
- **RetainedJob** : Nombre d'emplois que l'entreprise prévoit de conserver grâce au prêt.  <span style="color:green;">OK</span>
- **FranchiseCode** : Code indiquant si l'entreprise est une franchise (**00000** signifie que ce n'est pas une franchise).  <span style="color:green;">OK</span>
- **UrbanRural** : Indique si l'entreprise est située dans une zone **urbaine** (**1**), **rurale** (**2**), ou **non spécifiée** (**0**).  <span style="color:green;">OK</span>

## Conditions du prêt

- **RevLineCr** : Indique si le prêt est une ligne de crédit renouvelable <span style="color:green;">OK</span> 
- **LowDoc** : Indique si le prêt fait partie du programme *Low Documentation* <span style="color:green;">OK</span>

## Informations financières et remboursement

- **ChgOffDate** : Date à laquelle le prêt a été radié en cas de non-remboursement.  <span style="color:red;">Data Leaking</span>
- **DisbursementDate** : Date à laquelle les fonds ont été décaissés.  <span style="color:green;">OK</span>
- **DisbursementGross** : Montant total du prêt accordé.  <span style="color:green;">OK</span>
- **BalanceGross** : Montant restant dû sur le prêt.  <span style="color:red;">Data Leaking</span>
- **MIS_Status** : Statut du prêt :  <span style="color:blue;">TARGET</span>
  - **PIF (Paid In Full)** : Le prêt a été remboursé intégralement.  
  - **CHGOFF (Charged Off)** : Le prêt a été radié (perte pour la banque).  
- **ChgOffPrinGr** : Montant du principal du prêt qui a été radié (perdu par la banque).  <span style="color:red;">useless</span>
- **GrAppv** : Montant total approuvé par la banque.  <span style="color:green;">OK</span>
- **SBA_Appv** : Montant garanti par la *SBA* (une partie du prêt est garantie par l'agence).  <span style="color:green;">OK</span>


In [4]:
df_issue = df[df['MIS_Status'].isna() | (df['LowDoc'] == 0)]
df_issue.unique()

# df["RetainedJob"].unique()


AttributeError: 'DataFrame' object has no attribute 'unique'

In [96]:
# Nombre de zéros
num_zeros = (df["SBA_Appv"] == 0).sum()

# Nombre de valeurs NaN
num_nans = df["SBA_Appv"].isna().sum()

print(f"Nombre de zéros : {num_zeros}")
print(f"Nombre de NaN : {num_nans}")

df['SBA_Appv'].describe()


Nombre de zéros : 0
Nombre de NaN : 0


count          899164
unique          38326
top       $25,000.00 
freq            49579
Name: SBA_Appv, dtype: object

In [85]:
df["DisbursementDate"] = pd.to_datetime(df["DisbursementDate"], format="%d-%b-%y")
df["DisbursementDate"].describe() 

count                           896796
mean     2001-09-19 14:10:17.462165248
min                1969-05-22 00:00:00
25%                1997-05-31 00:00:00
50%                2002-12-31 00:00:00
75%                2006-03-31 00:00:00
max                2068-11-22 00:00:00
Name: DisbursementDate, dtype: object

compare 'DisbursementGross', 'GrAppv'

In [63]:
# On supprime les 14 lignes ou State = 0 ou null
df_clean = df[
    (df['State'] != 0) & (df['State'].notna()) &
    (df['BankState'] != 0) & (df['BankState'].notna()) &
    (df['NoEmp'] != 0) & (df['NoEmp'].notna()) &
    (df['NewExist'] != 0) & (df['NewExist'].notna())&
    (df['RetainedJob'].notna()) &
    (df['CreateJob'].notna()) &
    (df['UrbanRural'].notna()) &  #useless
    (df['RevLineCr'].notna())&
    (df['DisbursementDate'].notna())
]

df_clean['NAICS_2'] = df_clean['NAICS'].astype(str).str[:2]
# df_clean = df_clean[df_clean['NAICS_2'].notna() & (df_clean['NAICS_2'] != '0')]
df_clean["FranchiseCode"] = df["FranchiseCode"].where(df["FranchiseCode"] == 0, 1)
df_clean["RevLineCr"] = df["RevLineCr"].map(lambda x: 0 if x in ["N", "0"] else (1 if x in ["Y", "1"] else np.nan))
df_clean["LowDoc"] = df["LowDoc"].map(lambda x: 0 if x == "N" else (1 if x == "Y" else np.nan))
df_clean["DisbursementDate"] = pd.to_datetime(df["DisbursementDate"], format="%d-%b-%y")

columns_to_keep = ['State','BankState', 'NoEmp', 'NewExist', 'RetainedJob', 'CreateJob', 'UrbanRural', 'RevLineCr', 'DisbursementDate', 'DisbursementGross', 'GrAppv', 'SBA_Appv', 'MIS_Status', 'LowDoc']
# Vérifier le nombre de lignes après nettoyage
df_clean.shape[0]

/tmp/ipykernel_32708/4147023425.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['NAICS_2'] = df_clean['NAICS'].astype(str).str[:2]
/tmp/ipykernel_32708/4147023425.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["FranchiseCode"] = df["FranchiseCode"].where(df["FranchiseCode"] == 0, 1)
/tmp/ipykernel_32708/4147023425.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cav

870113

In [98]:
df2 = df_clean[columns_to_keep]
df2.head()

,NAICS,City,State,Zip,Bank,BankState,MIS_Status,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr
0,451120,EVANSVILLE,IN,47711,FIFTH THIRD BANK,OH,P I F,4,2.0,0,0,1,0,NaN
1,722410,NEW PARIS,IN,46526,1ST SOURCE BANK,IN,P I F,2,2.0,0,0,1,0,NaN
2,621210,BLOOMINGTON,IN,47401,GRANT COUNTY STATE BANK,IN,P I F,7,1.0,0,0,1,0,NaN
3,0,BROKEN ARROW,OK,74012,1ST NATL BK & TR CO OF BROKEN,OK,P I F,2,1.0,0,0,1,0,NaN
4,0,ORLANDO,FL,32801,FLORIDA BUS. DEVEL CORP,FL,P I F,14,1.0,7,7,1,0,NaN


In [ ]:
df2.isna().sum()

NAICS                 0
City                 27
State                 0
Zip                   0
Bank                  0
BankState             0
MIS_Status         1876
NoEmp                 0
NewExist              0
CreateJob             0
RetainedJob           0
FranchiseCode         0
UrbanRural            0
RevLineCr        870113
dtype: int64

In [51]:
df_zip_zero = df[df['Zip'] == 0]
df_zip_zero.shape[0]   # 283 lines

df_zip_zero = df[df['Zip'] == 0][['State', 'City']]

# Afficher les premières lignes des résultats
df_zip_zero.head()

,State,City
64,OR,Grass Valley
119,MO,Wappapello
7299,WA,WASHOUGAL
7693,CA,CORTE MADERA
13073,WV,FAYETTEVILLE


In [52]:
df_nulls = df[df["Zip"].isnull()]
print(df_nulls[["State", "Zip"]])

Empty DataFrame
Columns: [State, Zip]
Index: []


## Duplicates CHECK : OK

In [53]:
df[['ApprovalFY', 'ApprovalDate']]

,ApprovalFY,ApprovalDate
0,1997,28-Feb-97
1,1997,28-Feb-97
2,1997,28-Feb-97
3,1997,28-Feb-97
4,1997,28-Feb-97
...,...,...
899159,1997,27-Feb-97
899160,1997,27-Feb-97
899161,1997,27-Feb-97
899162,1997,27-Feb-97


In [54]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df[ "ChgOffPrinGr"][(df["MIS_Status"] == "P I F") & (df['ChgOffPrinGr'] != "$0.00 ")]

558        $3,330.00 
850       $10,270.00 
853       $97,486.00 
861        $2,310.00 
866        $3,783.00 
             ...     
895467     $6,462.00 
895669    $46,165.00 
897114    $20,878.00 
897735    $17,236.00 
898966    $30,002.00 
Name: ChgOffPrinGr, Length: 4884, dtype: object

# 1. PRE-PROCESSING

## 1.1. FEATURE SELECTION

In [56]:
# On filtre la db pour ne garder que les colonnes nécessaire à la régression logistique

# columns_to_keep = [
#     "NAICS", "City", "State", "Zip", "Bank", "BankState", "MIS_Status",
#     "NoEmp", "NewExist", "CreateJob", "RetainedJob",
#     "FranchiseCode", "UrbanRural", "RevLineCr"
# ]

filtered_df = df[columns_to_keep]
filtered_df.head(10)



,NAICS,City,State,Zip,Bank,BankState,MIS_Status,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr
0,451120,EVANSVILLE,IN,47711,FIFTH THIRD BANK,OH,P I F,4,2.0,0,0,1,0,0.0
1,722410,NEW PARIS,IN,46526,1ST SOURCE BANK,IN,P I F,2,2.0,0,0,1,0,0.0
2,621210,BLOOMINGTON,IN,47401,GRANT COUNTY STATE BANK,IN,P I F,7,1.0,0,0,1,0,0.0
3,0,BROKEN ARROW,OK,74012,1ST NATL BK & TR CO OF BROKEN,OK,P I F,2,1.0,0,0,1,0,0.0
4,0,ORLANDO,FL,32801,FLORIDA BUS. DEVEL CORP,FL,P I F,14,1.0,7,7,1,0,0.0
5,332721,PLAINVILLE,CT,6062,"TD BANK, NATIONAL ASSOCIATION",DE,P I F,19,1.0,0,0,1,0,0.0
6,0,UNION,NJ,7083,WELLS FARGO BANK NATL ASSOC,SD,CHGOFF,45,2.0,0,0,0,0,0.0
7,811118,SUMMERFIELD,FL,34491,REGIONS BANK,AL,P I F,1,2.0,0,0,1,0,0.0
8,721310,PORT SAINT JOE,FL,32456,CENTENNIAL BANK,FL,P I F,2,2.0,0,0,1,0,0.0
9,0,GLASTONBURY,CT,6073,WEBSTER BANK NATL ASSOC,CT,P I F,3,2.0,0,0,1,0,0.0


## 1.2. Formatage des données

In [57]:
# Supprimer le signe dollar ($) et les virgules dans la colonne "GrAppv"
# filtered_df['GrAppv'] = filtered_df['GrAppv'].replace({'\$': '', '': ''}, regex=True).astype(float)
# # filtered_df.head(3)
# print(f"type is :{type(filtered_df["GrAppv"])}")

## 1.3. Vérification des valeurs nulles

In [ ]:
filtered_df.isnull().sum()  # Vérifier les valeurs manquantes


NAICS                0
City                30
State               14
Zip                  0
Bank              1559
BankState         1566
MIS_Status        1997
NoEmp                0
NewExist           136
CreateJob            0
RetainedJob          0
FranchiseCode        0
UrbanRural           0
RevLineCr        19854
dtype: int64

## 1.4. INPUTING


In [59]:
print(type(45.000))


<class 'float'>
